# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [3]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    !cd ECE1508_GenAI && git pull

%cd ECE1508_GenAI

Cloning into 'ECE1508_GenAI'...
remote: Enumerating objects: 567, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 567 (delta 28), reused 63 (delta 21), pack-reused 488 (from 1)
Receiving objects: 100% (567/567), 19.69 MiB | 19.13 MiB/s, done.
Resolving deltas: 100% (229/229), done.
/content/ECE1508_GenAI


In [4]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 7.2 MB/s eta 0:00:00


In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [6]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: typeguard-4.5.2, anyio-4.14.2, langsmith-0.10.2
collected 20 items                                                             

steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [  5%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [ 10%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [ 15%]
steven/tests/test_data_pipeline.py::test_build_window_shapes_and_masks PASSED [ 20%]
steven/tests/test_data_pipeline.py::test_to_patchtst_input_patch_padding_mask PASSED [ 25%]
steven/tests/test_data_pipeline.py::test_window_sampler_unique_and_within_bounds PASSED [ 30%]
steven/tests/test_data_pipeline.py::test_window_sampler_respects_split_boundary PASSED [ 35%]

## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst.yaml)

Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first instead of the full config.

In [7]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst.yaml --device auto

03:02:25 device: cuda
03:02:25 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
03:02:25 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
03:02:34 epoch 1/20  train_loss=0.22820  val_loss=0.10943  (3.1s)
03:02:34   -> saved best checkpoint (val_loss=0.10943) to steven/outputs/patchtst_checkpoint.pt
03:02:36 epoch 2/20  train_loss=0.16943  val_loss=0.10375  (1.9s)
03:02:36   -> saved best checkpoint (val_loss=0.10375) to steven/outputs/patchtst_checkpoint.pt
03:02:38 epoch 3/20  train_loss=0.15020  val_loss=0.09948  (2.0s)
03:02:38   -> saved best checkpoint (val_loss=0.09948) to steven/outputs/patchtst_checkpoint.pt
03:02:40 epoch 4/20  train_loss=0.14227  val_loss=0.11023  (1.9s)
03:02:42 epoch 5/20  train_loss=0.14174  val_loss=0.10195  (1.9s)
03:02:44 epoch 6/20  train_loss=0.13829  val_loss=0.09548  (2.0s)
03:02:44   -> saved best checkpoint (val_loss=0.09548) to steven/outputs/patchtst_checkpoint.pt
03:02:46 epoch 7/20  train_

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae.yaml)

In [8]:
!python steven/src/train_cvae.py --config steven/configs/cvae.yaml --device auto

03:03:15 device: cuda
03:03:15 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
03:03:15 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
03:03:19 epoch 1/30  beta=0.20  train_loss=0.60684 (kl=0.8075)  val_loss=0.36387 (kl=0.8009)  (2.7s)
03:03:19   -> saved best checkpoint (val_loss=0.36387) to steven/outputs/cvae_checkpoint.pt
03:03:20 epoch 2/30  beta=0.40  train_loss=0.61963 (kl=0.8002)  val_loss=0.51419 (kl=0.8000)  (1.4s)
03:03:22 epoch 3/30  beta=0.60  train_loss=0.75839 (kl=0.8003)  val_loss=0.65516 (kl=0.8060)  (1.4s)
03:03:23 epoch 4/30  beta=0.80  train_loss=0.87611 (kl=0.8017)  val_loss=0.82204 (kl=0.8119)  (1.4s)
03:03:24 epoch 5/30  beta=1.00  train_loss=1.03131 (kl=0.8084)  val_loss=0.94145 (kl=0.8000)  (1.4s)
03:03:26 epoch 6/30  beta=1.00  train_loss=0.99197 (kl=0.8020)  val_loss=0.94482 (kl=0.8017)  (1.4s)
03:03:27 epoch 7/30  beta=1.00  train_loss=0.97670 (kl=0.8024)  val_loss=0.92643 (kl=0.8012)  (1.4s)
03:03:29

## Evaluate both models on the fixed test set

In [9]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

03:04:05 device: cuda
03:04:05 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
03:04:05 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
03:04:05 evaluating on 3000 fixed test windows
03:04:06 wrote metrics to steven/outputs/metrics.json
03:04:06 overall: {
  "n_windows": 3000,
  "patchtst_reparam_mae_rmse": [
    0.0975034162402153,
    0.3709149956703186
  ],
  "cvae_reparam_mae_rmse": [
    0.1871272772550583,
    0.5204401016235352
  ],
  "patchtst_ohlc_mae_rmse": [
    2.228653089731343,
    3.3902430110020076
  ],
  "cvae_ohlc_mae_rmse": [
    18.828935945535036,
    24.62711027237648
  ],
  "patchtst_volume_mae_rmse": [
    1988296.25,
    3500083.25
  ],
  "cvae_volume_mae_rmse": [
    3445505.0,
    4623668.5
  ],
  "patchtst_directional_accuracy": [
    0.521,
    0.5236666666666666,
    0.522
  ],
  "cvae_directional_accuracy": [
    0.5306666666666666,
    0.526,
    0.5153333333333333
  ],
  "cvae_avg_sample_variance"

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [ ]:
!python steven/src/update_report.py

## Pull results back down

Zips `steven/outputs/` (checkpoints, metrics.json, sample_plots) and downloads it -- or just `git add`/`commit`/`push` from here if you'd rather sync back through the repo.

In [10]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

  adding: steven/outputs/ (stored 0%)
  adding: steven/outputs/sample_plots/ (stored 0%)
  adding: steven/outputs/sample_plots/sample0_start24561_ctx70.png (deflated 13%)
  adding: steven/outputs/sample_plots/sample2_start26217_ctx28.png (deflated 13%)
  adding: steven/outputs/sample_plots/sample1_start26322_ctx63.png (deflated 7%)
  adding: steven/outputs/sample_plots/sample2_start24588_ctx35.png (deflated 6%)
  adding: steven/outputs/sample_plots/sample4_start26357_ctx14.png (deflated 6%)
  adding: steven/outputs/sample_plots/sample3_start25951_ctx28.png (deflated 7%)
  adding: steven/outputs/sample_plots/sample3_start25919_ctx49.png (deflated 13%)
  adding: steven/outputs/sample_plots/sample0_start26248_ctx21.png (deflated 7%)
  adding: steven/outputs/sample_plots/sample4_start26212_ctx49.png (deflated 13%)
  adding: steven/outputs/sample_plots/sample1_start24657_ctx42.png (deflated 13%)
  adding: steven/outputs/metrics.json (deflated 81%)
  adding: steven/outputs/cvae_checkpoint.pt

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>